<a href="https://colab.research.google.com/github/Thom-320/scres-ia/blob/main/notebooks/scresia_thesis_decision_ladder_UNIFIED_FINAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SCRESIA thesis-decision ladder — UNIFIED FINAL

This notebook unifies the two previous ladder runners:

- It keeps the **methodological contract** from the Codex definitive notebook.
- It keeps the **ready-to-run profiles** from the other session's overnight runner.

Scientific contract:

- **Strict Garrido-clean action contract:** `thesis_factorized = MultiDiscrete([6,3])`, one common inventory level `I_{t,S}` plus one shift level `S`.
- **Declared per-node extension:** `factorized = MultiDiscrete([6,6,6,3])` only when `inventory_period_mode="per_node"`.
- **Do not call `[6,6,6,3]` thesis-faithful.** It is a relaxation that tests whether different inventory periods at Op3, Op5, and Op9 add value.
- **DMLPA is not in this notebook.** DMLPA belongs in a later architecture-ablation notebook, after the static ladder is known.
- **Horizon caveat:** the default `max_steps=260` profiles are abbreviated weekly-decision experiments. Use them for ladder screening and PPO training/evaluation, not as the final thesis-replica horizon claim. Final Garrido-replica tables must state or rerun the thesis horizons: 10 years for R1/R2 and 20 years for R3.

Recommended overnight run: keep `RUN_PROFILE = "overnight_l1b"`.

In [1]:
# =====================
# 0) Run config
# =====================

# Recommended tonight: overnight_l1b
# Other useful profiles: smoke, l1a_today, l0_l1a_full, ppo_e2a_debug, ppo_e2a_serious, ppo_e2b_serious
RUN_PROFILE = "overnight_l1b"

GIT_URL = "https://github.com/Thom-320/scres-ia.git"
GIT_BRANCH = "main"
REPO_NAME = "scresia"

# Drive is useful for Colab overnight persistence, but this notebook falls back if mount fails.
USE_GOOGLE_DRIVE_OUTPUT = True
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/scresia_results/unified_thesis_ladder"
LOCAL_OUTPUT_DIR = "/content/unified_thesis_ladder"
KAGGLE_OUTPUT_DIR = "/kaggle/working/unified_thesis_ladder"

RUNTIME_PIP_PACKAGES = [
    "simpy>=4.1",
    "numpy>=1.26",
    "gymnasium>=0.29",
    "stable-baselines3>=2.3",
    "sb3-contrib>=2.3",
    "pandas>=2.2",
    "scipy>=1.10",
]

BASE_SEED = 42
REWARD_MODE = "ReT_cd_v1"
RISK_LEVEL = "increased"
OBSERVATION_VERSION = "v5"
OBSERVATION_MODE = "env_sdm_history_reward"
STEP_SIZE_HOURS = 168.0
STOCHASTIC_PT = True
GARRIDO_CFIS = "31-90"

PROFILE_CONFIG = {
    # Fast checks
    "smoke": dict(kind="static", levels=["L0_garrido", "L1a_uniform_IxS"], replications=1, garrido_cfis="31", max_steps=4),

    # Static ladder: thesis variables, explicit relaxations
    "l1a_today": dict(kind="static", levels=["L1a_uniform_IxS"], replications=30, garrido_cfis=GARRIDO_CFIS, max_steps=260),
    "l0_l1a_full": dict(kind="static", levels=["L0_garrido", "L1a_uniform_IxS"], replications=30, garrido_cfis=GARRIDO_CFIS, max_steps=260),
    "overnight_l1b": dict(kind="static", levels=["L1b_per_node_IxS"], replications=30, garrido_cfis=GARRIDO_CFIS, max_steps=260, l1b_screening_replications=5, l1b_top_k=20, l1b_top_replications=30),

    # Adaptive PPO after static ladder
    "ppo_e2a_debug": dict(kind="ppo", action_space_mode="thesis_factorized", inventory_period_mode="thesis_strict", timesteps=512, eval_episodes=1, seeds=[101], max_steps=8),
    "ppo_e2a_serious": dict(kind="ppo", action_space_mode="thesis_factorized", inventory_period_mode="thesis_strict", timesteps=200_000, eval_episodes=20, seeds=[101, 202, 303], max_steps=260),
    "ppo_e2b_serious": dict(kind="ppo", action_space_mode="factorized", inventory_period_mode="per_node", timesteps=200_000, eval_episodes=20, seeds=[101, 202, 303], max_steps=260),
}

if RUN_PROFILE not in PROFILE_CONFIG:
    raise ValueError(f"Unknown RUN_PROFILE={RUN_PROFILE!r}. Choose one of {sorted(PROFILE_CONFIG)}")

cfg = PROFILE_CONFIG[RUN_PROFILE]
print({"RUN_PROFILE": RUN_PROFILE, **cfg})

{'RUN_PROFILE': 'overnight_l1b', 'kind': 'static', 'levels': ['L1b_per_node_IxS'], 'replications': 30, 'garrido_cfis': '31-90', 'max_steps': 260, 'l1b_screening_replications': 5, 'l1b_top_k': 20, 'l1b_top_replications': 30}


## How to choose a profile

Use the profiles as a ladder:

1. `l1a_today`: common inventory period crossed with shift capacity, 18 static configurations.
2. `overnight_l1b`: per-node inventory periods crossed with shift capacity, 648 static configurations with screening and top-k confirmation.
3. `ppo_e2a_serious`: weekly adaptive PPO under the strict `[6,3]` thesis-decision contract.
4. `ppo_e2b_serious`: weekly adaptive PPO under the declared per-node extension `[6,6,6,3]`.

The adaptive PPO result must be compared against the best static policy in the same action space.

In [2]:
# =========================================
# 1) Portable Colab/Kaggle/local setup
# =========================================

import json
import os
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or Path("/content").exists()
IN_KAGGLE = Path("/kaggle/working").exists()


def run_cmd(cmd, cwd=None, check=True):
    print("$", " ".join(map(str, cmd)))
    result = subprocess.run(cmd, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    # Keep logs readable in notebooks but preserve enough error context.
    print(result.stdout[-6000:])
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with code {result.returncode}: {cmd}")
    return result

if IN_COLAB or IN_KAGGLE:
    run_cmd([sys.executable, "-m", "pip", "install", "-q", *RUNTIME_PIP_PACKAGES])
else:
    os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

if IN_COLAB:
    base_dir = Path("/content")
    repo_root = base_dir / REPO_NAME
    output_root = Path(LOCAL_OUTPUT_DIR)
    if USE_GOOGLE_DRIVE_OUTPUT:
        try:
            from google.colab import drive
            drive.mount("/content/drive")
            output_root = Path(DRIVE_OUTPUT_DIR)
        except Exception as exc:
            print(f"Drive mount failed; falling back to local output: {exc}")
            output_root = Path(LOCAL_OUTPUT_DIR)
elif IN_KAGGLE:
    base_dir = Path("/kaggle/working")
    repo_root = base_dir / REPO_NAME
    output_root = Path(KAGGLE_OUTPUT_DIR)
else:
    repo_root = Path.cwd()
    output_root = repo_root / "outputs" / "benchmarks" / "unified_thesis_ladder"

if IN_COLAB or IN_KAGGLE:
    shutil.rmtree(repo_root, ignore_errors=True)
    run_cmd(["git", "clone", "--depth", "1", "--branch", GIT_BRANCH, GIT_URL, str(repo_root)])

sys.path.insert(0, str(repo_root.parent))
sys.path.insert(0, str(repo_root))
output_root.mkdir(parents=True, exist_ok=True)
print("repo_root:", repo_root)
print("output_root:", output_root)
print("python path head:", sys.path[:3])

$ /usr/bin/python3 -m pip install -q simpy>=4.1 numpy>=1.26 gymnasium>=0.29 stable-baselines3>=2.3 sb3-contrib>=2.3 pandas>=2.2 scipy>=1.10
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.1/952.1 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.0/93.0 kB 3.7 MB/s eta 0:00:00

Mounted at /content/drive
$ git clone --depth 1 --branch main https://github.com/Thom-320/scres-ia.git /content/scresia
Cloning into '/content/scresia'...

repo_root: /content/scresia
output_root: /content/drive/MyDrive/scresia_results/unified_thesis_ladder
python path head: ['/content/scresia', '/content', '/content']


In [3]:
# ======================================
# 2) Verify required repo contracts
# ======================================

import pandas as pd

ppo_help = run_cmd([sys.executable, "scripts/run_thesis_decision_ppo_smoke.py", "--help"], cwd=repo_root).stdout
ladder_help = run_cmd([sys.executable, "scripts/run_thesis_decision_ladder.py", "--help"], cwd=repo_root).stdout

assert "thesis_factorized" in ppo_help, "PPO script lacks thesis_factorized action mode; pull latest main."
assert "--inventory-period-mode" in ppo_help, "PPO script lacks explicit inventory-period-mode; pull latest main."
assert "L0_garrido" in ladder_help, "Ladder script lacks L0_garrido."
assert "L1a_uniform_IxS" in ladder_help, "Ladder script lacks L1a_uniform_IxS."
assert "L1b_per_node_IxS" in ladder_help, "Ladder script lacks L1b_per_node_IxS."
print("contract check: ok")

$ /usr/bin/python3 scripts/run_thesis_decision_ppo_smoke.py --help
2026-06-12 03:24:35.754700: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-12 03:24:35.828943: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
usage: run_thesis_decision_ppo_smoke.py [-h] [--label LABEL]
                                        [--train-

In [4]:
# ==========================================
# 3) Run selected static ladder or PPO profile
# ==========================================

run_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
run_dirs = []

if cfg["kind"] == "static":
    run_label = f"{RUN_PROFILE}_{run_stamp}"
    cmd = [
        sys.executable,
        "scripts/run_thesis_decision_ladder.py",
        "--label", run_label,
        "--output-root", str(output_root / "static_ladder"),
        "--levels", *cfg["levels"],
        "--garrido-cfis", cfg["garrido_cfis"],
        "--replications", str(cfg["replications"]),
        "--reward-mode", REWARD_MODE,
        "--risk-level", RISK_LEVEL,
        "--observation-version", OBSERVATION_VERSION,
        "--observation-mode", OBSERVATION_MODE,
        "--step-size-hours", str(STEP_SIZE_HOURS),
        "--max-steps", str(cfg["max_steps"]),
    ]
    if STOCHASTIC_PT:
        cmd.append("--stochastic-pt")
    if "l1b_screening_replications" in cfg:
        cmd += ["--l1b-screening-replications", str(cfg["l1b_screening_replications"])]
    if "l1b_top_k" in cfg:
        cmd += ["--l1b-top-k", str(cfg["l1b_top_k"])]
    if "l1b_top_replications" in cfg:
        cmd += ["--l1b-top-replications", str(cfg["l1b_top_replications"])]
    run_cmd(cmd, cwd=repo_root)
    run_dirs.append(output_root / "static_ladder" / run_label)

elif cfg["kind"] == "ppo":
    for seed in cfg["seeds"]:
        run_label = f"{RUN_PROFILE}_seed_{seed}_{run_stamp}"
        cmd = [
            sys.executable,
            "scripts/run_thesis_decision_ppo_smoke.py",
            "--label", run_label,
            "--output-root", str(output_root / "ppo_adaptive"),
            "--algo", "ppo_mlp",
            "--action-space-mode", cfg["action_space_mode"],
            "--inventory-period-mode", cfg["inventory_period_mode"],
            "--observation-version", OBSERVATION_VERSION,
            "--observation-mode", OBSERVATION_MODE,
            "--reward-mode", REWARD_MODE,
            "--risk-level", RISK_LEVEL,
            "--step-size-hours", str(STEP_SIZE_HOURS),
            "--max-steps", str(cfg["max_steps"]),
            "--train-timesteps", str(cfg["timesteps"]),
            "--eval-episodes", str(cfg["eval_episodes"]),
            "--seed", str(seed),
            "--garrido-cfis", GARRIDO_CFIS,
            "--include-static-grid",
            "--eval-ai-on-garrido-cfis",
            "--learn-initial-decision",
            "--policy-net-arch", "medium",
        ]
        if STOCHASTIC_PT:
            cmd.append("--stochastic-pt")
        run_cmd(cmd, cwd=repo_root)
        run_dirs.append(output_root / "ppo_adaptive" / run_label)
else:
    raise ValueError(f"Unknown profile kind: {cfg['kind']}")

print("run_dirs:")
for path in run_dirs:
    print(" -", path)

$ /usr/bin/python3 scripts/run_thesis_decision_ladder.py --label overnight_l1b_20260612T032443Z --output-root /content/drive/MyDrive/scresia_results/unified_thesis_ladder/static_ladder --levels L1b_per_node_IxS --garrido-cfis 31-90 --replications 30 --reward-mode ReT_cd_v1 --risk-level increased --observation-version v5 --observation-mode env_sdm_history_reward --step-size-hours 168.0 --max-steps 260 --stochastic-pt --l1b-screening-replications 5 --l1b-top-k 20 --l1b-top-replications 30
{
  "policy": "L1b_per_node_I168_I336_I672_S2",
  "ladder_level": "L1b_per_node_IxS",
  "stage": "screening",
  "episode_count": 5,
  "reward_total_mean": 144.36065401413026,
  "fill_rate_order_level_mean": 0.84745122718691,
  "order_level_ret_mean": 0.23453470126941625,
  "assembly_shift_hours_mean": 87360.0,
  "inventory_target_total_mean": 109080.0,
  "pending_backorders_count_mean": 58.2,
  "pending_backorder_qty_mean": 150719.4
}
Saved to: /content/drive/MyDrive/scresia_results/unified_thesis_ladde

In [5]:
# ===========================
# 4) Read result summaries
# ===========================

summary_rows = []

for run_dir in run_dirs:
    print("\n===", run_dir, "===")
    summary_json = run_dir / "summary.json"
    policy_csv = run_dir / "policy_summary.csv"
    eval_csv = run_dir / "evaluation_summary.csv"

    if summary_json.exists():
        payload = json.loads(summary_json.read_text())
        interesting = {k: payload.get(k) for k in [
            "label", "levels", "replications", "max_steps", "best_policy_by_fill_rate", "action_space_mode", "inventory_period_mode", "action_dim"
        ] if k in payload}
        print(json.dumps(interesting, indent=2))

    if policy_csv.exists():
        df = pd.read_csv(policy_csv)
        display(df.sort_values(
            ["fill_rate_order_level_mean", "order_level_ret_mean", "reward_total_mean"],
            ascending=[False, False, False],
        ).head(25))
        if "ladder_level" in df.columns:
            display(df.groupby("ladder_level")[["fill_rate_order_level_mean", "order_level_ret_mean", "reward_total_mean"]].max())

    if eval_csv.exists():
        df = pd.read_csv(eval_csv)
        sort_cols = [c for c in ["fill_rate_order_level_mean", "order_level_ret_mean", "reward_total_mean"] if c in df.columns]
        display(df.sort_values(sort_cols, ascending=[False] * len(sort_cols)).head(25))

print("Done.")


=== /content/drive/MyDrive/scresia_results/unified_thesis_ladder/static_ladder/overnight_l1b_20260612T032443Z ===
{
  "levels": [
    "L1b_per_node_IxS"
  ],
  "replications": 30,
  "max_steps": 260,
  "best_policy_by_fill_rate": {
    "assembly_shift_hours_mean": 87360.0,
    "episode_count": 5,
    "fill_rate_order_level_mean": 0.84745122718691,
    "inventory_target_total_mean": 109080.0,
    "ladder_level": "L1b_per_node_IxS",
    "order_level_ret_mean": 0.23453470126941625,
    "pending_backorder_qty_mean": 150719.4,
    "pending_backorders_count_mean": 58.2,
    "policy": "L1b_per_node_I168_I336_I672_S2",
    "reward_total_mean": 144.36065401413026,
    "stage": "screening"
  }
}


,policy,ladder_level,stage,episode_count,reward_total_mean,fill_rate_order_level_mean,order_level_ret_mean,assembly_shift_hours_mean,inventory_target_total_mean,pending_backorders_count_mean,pending_backorder_qty_mean
294,L1b_per_node_I168_I336_I672_S2,L1b_per_node_IxS,screening,5,144.360654,0.847451,0.234535,87360.0,109080.0,58.2,150719.4
6,L1b_per_node_I0_I0_I168_S1,L1b_per_node_IxS,screening,5,143.670021,0.847103,0.227066,43680.0,15750.0,58.6,150733.8
343,L1b_per_node_I336_I0_I336_S1,L1b_per_node_IxS,screening,5,145.279305,0.846822,0.227345,43680.0,62220.0,58.8,153670.8
32,L1b_per_node_I0_I1344_I504_S2,L1b_per_node_IxS,screening,5,143.518903,0.843046,0.225244,87360.0,170130.0,58.4,151555.0
636,L1b_per_node_I672_I504_I168_S1,L1b_per_node_IxS,screening,5,142.380334,0.842920,0.220923,43680.0,123270.0,58.4,151193.4
320,L1b_per_node_I168_I672_I1344_S3,L1b_per_node_IxS,screening,5,142.006559,0.842417,0.235473,131040.0,202800.0,56.8,145674.0
96,L1b_per_node_I0_I672_I1344_S1,L1b_per_node_IxS,screening,5,141.061916,0.841410,0.231267,43680.0,187440.0,57.6,147487.8
442,L1b_per_node_I336_I672_I504_S3,L1b_per_node_IxS,screening,5,140.555680,0.840780,0.224325,131040.0,139410.0,57.6,147637.2
599,L1b_per_node_I672_I168_I168_S1,L1b_per_node_IxS,screening,5,138.222397,0.840780,0.222363,43680.0,92550.0,57.8,148266.4
566,L1b_per_node_I672_I0_I336_S2,L1b_per_node_IxS,screening,5,140.158606,0.840025,0.232793,87360.0,92940.0,57.0,145939.6


,fill_rate_order_level_mean,order_level_ret_mean,reward_total_mean
ladder_level,,,
L1b_per_node_IxS,0.847451,0.235473,145.279305


Done.


## Reporting rule

Use this language in the paper:

> The action variables use the thesis decision variable families and values. The strict Garrido-clean action contract uses a common inventory level and shift level. We then relax the thesis protocol in declared steps: crossing inventory with capacity, allowing per-node inventory periods, and allowing weekly adaptive re-decision. Each adaptive result is compared against the best static policy in the same action space.

Do not write that `MultiDiscrete([6,6,6,3])` is 1:1 with Garrido. It is the per-node extension.